# Penguins FLAML AutoML Classification + Stratified K‑Fold

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score, confusion_matrix, classification_report
from flaml import AutoML

In [2]:
# --- Load ---
csv_path = "./data/penguins_synthetic_5000.csv"  # change if needed
df = pd.read_csv(csv_path).dropna().reset_index(drop=True)


In [3]:

target_col = "species"
feature_cols = [c for c in df.columns if c != target_col]

In [4]:
# --- Encode ---
X = pd.get_dummies(df[feature_cols], drop_first=True)
le = LabelEncoder()
y = le.fit_transform(df[target_col].astype(str))

In [5]:
# --- Stratified K‑Fold with FLAML ---
SEED = 42
N_FOLDS = 7
TIME_BUDGET_FOLD = 60  # seconds per fold

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_pred = np.empty_like(y)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[val_idx:=va_idx]  # keep a name for clarity

    automl = AutoML()
    automl.fit(
        X_train=X_tr, y_train=y_tr,
        task="classification", metric="accuracy",
        time_budget=TIME_BUDGET_FOLD, seed=SEED+fold, verbose=0
    )

    y_pred = automl.predict(X_va)
    oof_pred[va_idx] = y_pred

    acc = accuracy_score(y_va, y_pred)
    f1m = f1_score(y_va, y_pred, average="macro")
    bal = balanced_accuracy_score(y_va, y_pred)
    print(f"[Fold {fold}] Acc={acc:.4f} | F1-macro={f1m:.4f} | BalAcc={bal:.4f}")

[Fold 1] Acc=1.0000 | F1-macro=1.0000 | BalAcc=1.0000
[Fold 2] Acc=1.0000 | F1-macro=1.0000 | BalAcc=1.0000
[Fold 3] Acc=1.0000 | F1-macro=1.0000 | BalAcc=1.0000
[Fold 4] Acc=1.0000 | F1-macro=1.0000 | BalAcc=1.0000
[Fold 5] Acc=1.0000 | F1-macro=1.0000 | BalAcc=1.0000
[Fold 6] Acc=1.0000 | F1-macro=1.0000 | BalAcc=1.0000
[Fold 7] Acc=1.0000 | F1-macro=1.0000 | BalAcc=1.0000


In [6]:
# --- OOF summary ---
print("\n=== OOF Performance ===")
print("Accuracy:", accuracy_score(y, oof_pred))
print("F1-macro:", f1_score(y, oof_pred, average="macro"))
print("Balanced Accuracy:", balanced_accuracy_score(y, oof_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y, oof_pred))

print("\nClassification Report:")
print(classification_report(y, oof_pred, target_names=le.inverse_transform(np.arange(len(le.classes_)))))


=== OOF Performance ===
Accuracy: 1.0
F1-macro: 1.0
Balanced Accuracy: 1.0

Confusion Matrix:
[[2135    0    0]
 [   0  994    0]
 [   0    0 1740]]

Classification Report:
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00      2135
   Chinstrap       1.00      1.00      1.00       994
      Gentoo       1.00      1.00      1.00      1740

    accuracy                           1.00      4869
   macro avg       1.00      1.00      1.00      4869
weighted avg       1.00      1.00      1.00      4869

